**MGMT298D: Science and Strategy of AI**

# Week 6: Building a Tiny LLM

In this notebook, we build a small GPT-style language model and train it on Yelp reviews.

The goal is to see the core mechanics of next-token prediction: tokenize text, train a causal transformer, and compare its generated text after 1, 2, and 5 epochs.

# 1 Setup

In [ ]:
#@title Import libraries { display-mode: "form" }
import os
import logging
import warnings

# Keep Hugging Face public-dataset downloads quiet in Colab.
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)

warnings.filterwarnings("ignore", message=r".*HF_TOKEN.*", category=UserWarning)
warnings.filterwarnings("ignore", message=r".*unauthenticated requests.*", category=UserWarning)
warnings.filterwarnings("ignore", module=r"huggingface_hub.*")

import numpy as np
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset, disable_progress_bar
from huggingface_hub.utils import disable_progress_bars

disable_progress_bar()
disable_progress_bars()

import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, HTML

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


---
# 2 Load Yelp Reviews

The full Yelp Review Full training split contains 650,000 reviews. This notebook uses 500,000 reviews for broader language exposure while keeping training manageable in Colab.

In [ ]:
dataset = load_dataset("yelp_review_full", split="train", token=False)

FULL_TRAIN_SIZE = len(dataset)
NUM_REVIEWS = 500000

texts = dataset["text"][:NUM_REVIEWS]

print(f"Using {len(texts):,} Yelp reviews from a training split of {FULL_TRAIN_SIZE:,}.")

In [ ]:
#@title Tokenize reviews and build training sequences { display-mode: "form" }
VOCAB_SIZE = 25000
SEQ_LEN = 64   # context window

# Convert raw text into integer token IDs.
vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=SEQ_LEN + 1
)
vectorizer.adapt(texts)
vocab = vectorizer.get_vocabulary()

# Vectorize the review text.
all_tokens = vectorizer(np.array(texts)).numpy()

# Drop very short reviews so each training example has enough context.
all_tokens = all_tokens[np.sum(all_tokens > 0, axis=1) > 20]

# Input = tokens[:-1]. Target = tokens shifted one position to the right.
# This trains the model to predict the next token at every position.
x_all = all_tokens[:, :-1]
y_all = all_tokens[:, 1:]

# Use a fixed random split so validation is representative and reproducible.
VALIDATION_FRACTION = 0.20
rng = np.random.default_rng(seed=42)
indices = rng.permutation(len(x_all))
val_size = int(len(x_all) * VALIDATION_FRACTION)
val_idx = indices[:val_size]
train_idx = indices[val_size:]

x_train, y_train = x_all[train_idx], y_all[train_idx]
x_val, y_val = x_all[val_idx], y_all[val_idx]

print(f"Training examples: {len(x_train):,} | Validation examples: {len(x_val):,}")


---
# 3 Build a Tiny GPT-Style Language Model

This model is a small decoder-only transformer. It is much smaller than production LLMs, but it uses the same basic next-token prediction idea.

Architecture:

1. **Token embeddings** convert token IDs into vectors.
2. **Position embeddings** add information about where each token appears.
3. **Causal self-attention** lets each token use earlier tokens while preventing it from seeing future tokens.
4. **Feed-forward layers** process the representation at each position.
5. **Residual connections and layer normalization** help stabilize training.
6. **The output layer** predicts a probability distribution over the vocabulary for the next token.

The model is built directly with Keras layers so the main pieces are visible in one cell.

In [ ]:
#@title Model settings { display-mode: "form" }
VOCAB_SIZE = 25000
SEQ_LEN = 64       # number of tokens the model can look back at
EMBED_DIM = 128    # size of each token vector
NUM_HEADS = 4      # number of attention heads per transformer block
FF_DIM = 256       # hidden size of the feed-forward network
NUM_BLOCKS = 2     # number of transformer blocks below

In [ ]:
#@title Build the model manually: embeddings, causal attention, feed-forward layers { display-mode: "form" }
# We build a tiny decoder-only transformer directly with Keras layers.
# This keeps the architecture visible instead of hiding it inside a custom class.

inputs = layers.Input(shape=(SEQ_LEN,), name="tokens")

# -------------------------------------------------------------------
# 1. Token embeddings
# -------------------------------------------------------------------
# Each token ID is mapped to a dense vector of length EMBED_DIM.
# Shape: (batch, sequence length) -> (batch, sequence length, embedding dim)
token_embeddings = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    name="token_embedding"
)(inputs)

# -------------------------------------------------------------------
# 2. Position embeddings
# -------------------------------------------------------------------
# Transformers do not know word order by default. We add one learned
# position vector for each location in the context window.
positions = tf.range(start=0, limit=SEQ_LEN, delta=1)
position_embeddings = layers.Embedding(
    input_dim=SEQ_LEN,
    output_dim=EMBED_DIM,
    name="position_embedding"
)(positions)

# The model representation starts as token meaning + token position.
x = token_embeddings + position_embeddings

# -------------------------------------------------------------------
# 3. Transformer block 1
# -------------------------------------------------------------------
# Causal self-attention lets each position look only at current and earlier
# positions. This prevents the model from peeking at the next token.
attn_1 = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBED_DIM // NUM_HEADS,
    name="causal_attention_1"
)(x, x, use_causal_mask=True)

# Residual connection + layer normalization.
x = layers.LayerNormalization(name="norm_1")(x + attn_1)

# Feed-forward network applied independently at each position.
ff_1 = layers.Dense(FF_DIM, activation="relu", name="ff_1_dense_1")(x)
ff_1 = layers.Dense(EMBED_DIM, name="ff_1_dense_2")(ff_1)

# Second residual connection + layer normalization.
x = layers.LayerNormalization(name="norm_2")(x + ff_1)

# -------------------------------------------------------------------
# 4. Transformer block 2
# -------------------------------------------------------------------
# A second block gives the model another round of contextual mixing.
attn_2 = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBED_DIM // NUM_HEADS,
    name="causal_attention_2"
)(x, x, use_causal_mask=True)

x = layers.LayerNormalization(name="norm_3")(x + attn_2)

ff_2 = layers.Dense(FF_DIM, activation="relu", name="ff_2_dense_1")(x)
ff_2 = layers.Dense(EMBED_DIM, name="ff_2_dense_2")(ff_2)

x = layers.LayerNormalization(name="norm_4")(x + ff_2)

# -------------------------------------------------------------------
# 5. Next-token prediction head
# -------------------------------------------------------------------
# For every position, predict a probability distribution over the vocabulary.
outputs = layers.Dense(VOCAB_SIZE, activation="softmax", name="next_token_probs")(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
#@title Compact model overview { display-mode: "form" }
print(
    f"Model: {NUM_BLOCKS} decoder-only transformer blocks | "
    f"context length: {SEQ_LEN} tokens | "
    f"vocabulary: {VOCAB_SIZE:,} tokens | "
    f"embedding dim: {EMBED_DIM} | "
    f"attention heads: {NUM_HEADS} | "
    f"feed-forward dim: {FF_DIM} | "
    f"trainable parameters: {model.count_params():,}"
)

---
# 4 Generation Helpers

For each prompt, the widget shows:

1. the model's immediate next-token distribution, and
2. a 20-token generated continuation.

In [ ]:
#@title Define generation utilities + interactive widget { display-mode: "form" }
id_to_word = dict(enumerate(vocab))


def _prompt_tokens(prompt):
    tokens = vectorizer([prompt]).numpy()[0]
    nonzero = np.where(tokens > 0)[0]
    if len(nonzero) == 0:
        return []
    return list(tokens[:nonzero[-1] + 1])


def next_token_probs(model, prompt):
    out = _prompt_tokens(prompt)
    if not out:
        return None

    padded = np.zeros(SEQ_LEN, dtype="int32")
    context = out[-SEQ_LEN:]
    padded[:len(context)] = context
    pred_pos = len(context) - 1

    probs = model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]
    probs = probs.copy()

    # Do not display or sample padding or unknown tokens.
    probs[0] = 0
    probs[1] = 0
    probs = probs / probs.sum()
    return probs


def plot_next_token_distribution(model, prompt, top_k=10):
    probs = next_token_probs(model, prompt)
    if probs is None:
        print("Type a prompt first.")
        return

    top_ids = np.argsort(probs)[-top_k:][::-1]
    words = [id_to_word[i] for i in top_ids]
    values = [probs[i] for i in top_ids]

    plt.figure(figsize=(6, 2.4))
    plt.bar(words, values)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Probability")
    plt.title("Immediate next-token distribution")
    plt.tight_layout()
    plt.show()


def generate_text(model, prompt, length=20):
    out = _prompt_tokens(prompt)
    if not out:
        return "(type a prompt first)"

    prompt_len = len(out)

    for _ in range(length):
        padded = np.zeros(SEQ_LEN, dtype="int32")
        context = out[-SEQ_LEN:]
        padded[:len(context)] = context
        pred_pos = len(context) - 1

        probs = model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]
        probs = probs.copy()
        probs[0] = 0
        probs[1] = 0
        probs = probs / probs.sum()

        next_id = np.random.choice(len(probs), p=probs)
        out.append(next_id)

    prompt_words = [id_to_word[t] for t in out[:prompt_len]]
    generated_words = [id_to_word[t] for t in out[prompt_len:]]
    return f"<b>{' '.join(prompt_words)}</b> {' '.join(generated_words)}"


def make_generator_widget(model, title="Try it yourself"):
    prompt_box = widgets.Text(
        value="the food was",
        placeholder="Type a prompt...",
        description="Prompt:",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "60px"}
    )

    button = widgets.Button(description="Generate", button_style="primary")

    chart_output = widgets.Output()
    text_output = widgets.Output(layout=widgets.Layout(min_height="50px", padding="8px"))

    def run_generation(_=None):
        with chart_output:
            chart_output.clear_output(wait=True)
            plot_next_token_distribution(model, prompt_box.value, top_k=10)

        with text_output:
            text_output.clear_output(wait=True)
            generated = generate_text(model, prompt_box.value, length=20)
            display(HTML(f"<div style='font-size:15px'>{generated}</div>"))

    button.on_click(run_generation)

    try:
        prompt_box.on_submit(run_generation)
    except Exception:
        pass

    display(HTML(f"<h4>{title}</h4>"))
    display(widgets.HBox([prompt_box, button]))
    display(HTML("<b>Immediate next-token distribution</b>"))
    display(chart_output)
    display(HTML("<b>Generated continuation: next 20 tokens</b>"))
    display(text_output)

    run_generation()

---
# 5 Phase 1 — Train for 1 Epoch

In [ ]:
h1 = model.fit(x_train, y_train, batch_size=128, epochs=1, validation_data=(x_val, y_val))

all_loss = list(h1.history['loss'])
all_val  = list(h1.history['val_loss'])
all_acc  = list(h1.history['accuracy'])

In [ ]:
make_generator_widget(model, "Phase 1 — Generate after 1 epoch")

---
# 6 Phase 2 — Train to 2 Total Epochs

In [ ]:
h2 = model.fit(x_train, y_train, batch_size=128, epochs=1, validation_data=(x_val, y_val))

all_loss += h2.history['loss']
all_val  += h2.history['val_loss']
all_acc  += h2.history['accuracy']

In [ ]:
make_generator_widget(model, "Phase 2 — Generate after 2 epochs")

---
# 7 Phase 3 — Train to 5 Total Epochs

In [ ]:
h3 = model.fit(x_train, y_train, batch_size=128, epochs=3, validation_data=(x_val, y_val))

all_loss += h3.history['loss']
all_val  += h3.history['val_loss']
all_acc  += h3.history['accuracy']

In [ ]:
make_generator_widget(model, "Phase 3 — Generate after 5 epochs")

---
# 8 Training Progress

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, len(all_loss) + 1)

ax1.plot(epochs, all_loss, 'b-o', ms=3, label='Train')
ax1.plot(epochs, all_val, 'r-o', ms=3, label='Val')
ax1.axvline(1, color='gray', ls='--', alpha=.5)
ax1.axvline(2, color='gray', ls='--', alpha=.5)
ax1.axvline(5, color='gray', ls='--', alpha=.5)
ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss (lower = better predictions)')
ax1.legend()

ax2.plot(epochs, all_acc, 'b-o', ms=3)
ax2.axvline(1, color='gray', ls='--', alpha=.5)
ax2.axvline(2, color='gray', ls='--', alpha=.5)
ax2.axvline(5, color='gray', ls='--', alpha=.5)
ax2.set(xlabel='Epoch', ylabel='Accuracy', title='Next-Token Prediction Accuracy')

plt.tight_layout()
plt.show()